# Lab 2 — FER-2013: kiểm tra kết quả đã huấn luyện

Notebook này ưu tiên kiểm tra kết quả Google Colab đã được tích hợp vào `results/` và `weights/`. **Không cần train lại** để sử dụng project. Nếu muốn huấn luyện lại, chỉ chạy các cell được đánh dấu tùy chọn ở cuối notebook.

## 1. Kiểm tra dataset

FER-2013 dùng ảnh grayscale 48×48 với 7 lớp. Tập train được chia stratified 90%/10% cho train/validation trong script; tập test được giữ riêng.

In [ ]:
from pathlib import Path
import json
DATA = Path('data/fer2013')
assert (DATA/'train').is_dir() and (DATA/'test').is_dir(), 'Không thấy data/fer2013; kiểm tra cấu trúc project.'
for split in ['train', 'test']:
    counts = {p.name: len([x for x in p.iterdir() if x.is_file()]) for p in (DATA/split).iterdir() if p.is_dir()}
    print(split, sum(counts.values()), counts)


## 2. Đọc metric đã có

Các giá trị được đọc từ file kết quả, không tự tạo lại hoặc ước đoán.

In [ ]:
from IPython.display import display, Image
for name in ['cnn', 'transfer']:
    folder = Path('results')/name
    metrics = json.loads((folder/'metrics.json').read_text(encoding='utf-8'))
    print(name, {k: metrics[k] for k in ['test_accuracy','test_macro_f1','parameters','inference_fps_batch_estimate','device']})
    display(Image(filename=str(folder/'training_curves.png')))
    display(Image(filename=str(folder/'confusion_matrix.png')))
    print((folder/'classification_report.txt').read_text(encoding='utf-8'))


## 3. Kiểm tra checkpoint và ví dụ dự đoán

Lệnh sau chạy demo trên ảnh FER-2013 mẫu `test.jpg`. Vì đây là ảnh crop nhỏ 48×48, detector có fallback dùng toàn ảnh làm vùng mặt nếu Haar Cascade không phát hiện được.

In [ ]:
!python src/app.py --source test.jpg --model transfer --no-display


## 4. Huấn luyện lại (tùy chọn, không cần cho lần chạy hiện tại)

Chỉ chạy các lệnh dưới đây nếu bạn chủ động muốn huấn luyện lại và có đủ thời gian/tài nguyên. Chạy sẽ ghi đè checkpoint/kết quả tương ứng.

In [ ]:
# %run src/train.py --data data/fer2013 --model cnn --epochs 20 --batch-size 64
# %run src/train.py --data data/fer2013 --model transfer --epochs 10 --batch-size 32
